# Yahoo!ニュース：『熊本地震』の記事収集

Yahoo!ニュース検索の9カテゴリー（国内・国際・経済・エンタメ・スポーツ・IT・科学・ライフ・地域）を50件ずつたどり、**2026年7月28日 00:00（日本時間）以降**の記事を探します。URLをキーにmaster Excelと比較し、master未登録の記事だけを合計最大1,000件取得して末尾へ追加します。記事ページから本文と正確な公開日時を取得し、本文を取得できない場合だけ検索結果の抜粋を保存します。

> 実行前にYahoo! JAPANの利用規約・robots.txt・配信元記事の権利を確認し、許可された範囲で使用してください。アクセス間隔は `REQUEST_INTERVAL_SECONDS` で調整できます。

In [1]:
# 未導入の場合だけ、先頭の # を外して一度実行してください。
# %pip install requests beautifulsoup4 pandas openpyxl


In [2]:
import json
import re
import time
from copy import copy
from datetime import datetime
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
from zoneinfo import ZoneInfo

import pandas as pd
import requests
from bs4 import BeautifulSoup
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

KEYWORD = "熊本地震"
JST = ZoneInfo("Asia/Tokyo")
CUTOFF = datetime(2026, 7, 28, 0, 0, tzinfo=JST)  # この日時を含む
SEARCH_API_URL = "https://news.yahoo.co.jp/api/public/search-feed"
RESULTS_PER_PAGE = 50
MAX_SEARCH_START = 1000  # Yahoo側はstart=1001以上を500エラーにする
MAX_NEW_ARTICLES = 1000  # 1回の実行でmasterへ追加する新規記事の上限
SEARCH_CATEGORIES = {
    "domestic": "国内",
    "world": "国際",
    "business": "経済",
    "entertainment": "エンタメ",
    "sports": "スポーツ",
    "it": "IT",
    "science": "科学",
    "life": "ライフ",
    "local": "地域",
}
REQUEST_INTERVAL_SECONDS = 1.2
REQUEST_TIMEOUT_SECONDS = 30

MASTER_PATH = Path("/Users/tj/IdeaProjects/sample/get-news/yahoo_熊本地震_20260728以降_master.xlsx")
if not MASTER_PATH.exists():
    raise FileNotFoundError(f"master Excelが見つかりません: {MASTER_PATH}")

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0 Safari/537.36"
    ),
    "Accept-Language": "ja,en-US;q=0.8,en;q=0.6",
})
retry = Retry(
    total=4, connect=4, read=4, backoff_factor=1.0,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({"GET"}),
)
session.mount("https://", HTTPAdapter(max_retries=retry))

print(f"検索語: {KEYWORD}")
print(f"対象期間: {CUTOFF.isoformat()} 以降")
print(f"更新対象: {MASTER_PATH.resolve()}")
print(f"1回の新規取得上限: {MAX_NEW_ARTICLES:,}件")


検索語: 熊本地震
対象期間: 2026-07-28T00:00:00+09:00 以降
更新対象: /Users/tj/IdeaProjects/sample/get-news/yahoo_熊本地震_20260728以降_master.xlsx
1回の新規取得上限: 1,000件


In [3]:
def clean_text(value):
    """検索結果の強調表示用制御文字と余分な空白を除く。"""
    if value is None:
        return ""
    text = str(value).replace("\x02", "").replace("\x03", "")
    text = text.replace("\u3000", " ").replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def canonical_yahoo_url(url):
    parts = urlsplit(url)
    return urlunsplit((parts.scheme, parts.netloc, parts.path.rstrip("/"), "", ""))


def parse_search_datetime(publish_time):
    """検索結果の YYYY/M/D または M/D(曜) と時刻をJSTのdatetimeにする。"""
    date_text = str((publish_time or {}).get("date") or "")
    time_text = str((publish_time or {}).get("time") or "00:00")
    full_date_match = re.search(r"(?<!\d)(\d{4})/(\d{1,2})/(\d{1,2})(?!\d)", date_text)
    short_date_match = re.search(r"(?<![\d/])(\d{1,2})/(\d{1,2})(?!\d)", date_text)
    time_match = re.search(r"(?<!\d)(\d{1,2}):(\d{2})(?!\d)", time_text)

    if full_date_match:
        year, month, day = map(int, full_date_match.groups())
    elif short_date_match:
        year = CUTOFF.year
        month, day = map(int, short_date_match.groups())
    else:
        return None

    hour, minute = map(int, time_match.groups()) if time_match else (0, 0)
    try:
        return datetime(year, month, day, hour, minute, tzinfo=JST)
    except ValueError:
        # Yahoo側に想定外の表示があっても、検索全体は停止させない。
        return None


def fetch_search_candidates(keyword, cutoff):
    """9カテゴリーを個別に巡回し、各カテゴリー最大1,000件から期間内候補を集める。"""
    candidates = []
    seen_urls = set()
    total_results = 0
    category_stats = []

    for category_id, category_name in SEARCH_CATEGORIES.items():
        start = 1
        category_total = None
        category_scanned = 0
        category_added = 0
        reached_cutoff = False
        reached_limit = False
        print(f"\n===== {category_name} の検索を開始 =====")

        while start <= MAX_SEARCH_START:
            params = {
                "start": start, "query": keyword, "categories": category_id,
                "qrw": "true", "results": RESULTS_PER_PAGE,
                "ei": "utf-8", "source": "other",
            }
            try:
                response = session.get(SEARCH_API_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
                response.raise_for_status()
                data = response.json()
            except (requests.RequestException, ValueError) as exc:
                print(f"[{category_name}] 検索取得に失敗: start={start:,} / {exc!r}")
                print("このカテゴリーは取得済みの分まで保存対象にし、次へ進みます。")
                break

            if category_total is None:
                category_total = int(data.get("totalResults") or 0)
                total_results += category_total
            contents = data.get("contents") or []
            page_dates = []
            category_scanned += sum(1 for item in contents if item.get("viewType") == 0)

            for item in contents:
                if item.get("viewType") != 0 or not item.get("permalink"):
                    continue
                url = canonical_yahoo_url(item["permalink"])
                published = parse_search_datetime(item.get("publishTime"))
                if published is not None:
                    page_dates.append(published)
                if published is not None and published < cutoff:
                    continue
                if url in seen_urls:
                    continue
                highlight = item.get("highlightSearchText") or {}
                candidates.append({
                    "title": clean_text(highlight.get("headline")),
                    "snippet": clean_text(highlight.get("body")),
                    "published_at_search": published.isoformat() if published else "",
                    "url": url,
                    "category": category_name,
                    "source": clean_text((item.get("media") or {}).get("name")),
                })
                seen_urls.add(url)
                category_added += 1

            print(
                f"[{category_name}] {start:,}〜{start + len(contents) - 1:,} / "
                f"ヒット {category_total:,}件、期間内追加 {category_added:,}件"
            )

            # 各カテゴリー内も新着順。境界日より古い記事を含むページで終了。
            if page_dates and min(page_dates) < cutoff:
                reached_cutoff = True
                break
            next_start = data.get("nextStart")
            if next_start and next_start > MAX_SEARCH_START:
                reached_limit = True
                print(f"[{category_name}] Yahoo側の1,000件上限に到達しました。")
                break
            if data.get("isLimit") or not contents or not next_start or next_start <= start:
                break
            start = next_start
            time.sleep(REQUEST_INTERVAL_SECONDS)

        category_stats.append({
            "カテゴリー": category_name,
            "検索ヒット数": category_total or 0,
            "確認件数": category_scanned,
            "期間内追加件数": category_added,
            "期間境界到達": reached_cutoff,
            "1000件上限到達": reached_limit,
        })

    print("\n===== カテゴリー別検索結果 =====")
    display(pd.DataFrame(category_stats))
    return candidates, total_results


In [4]:
def iter_jsonld_objects(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from iter_jsonld_objects(child)
    elif isinstance(value, list):
        for child in value:
            yield from iter_jsonld_objects(child)


def get_newsarticle_jsonld(soup):
    for script in soup.select('script[type="application/ld+json"]'):
        try:
            value = json.loads(script.string or script.get_text())
        except (TypeError, json.JSONDecodeError):
            continue
        for obj in iter_jsonld_objects(value):
            object_type = obj.get("@type")
            types = object_type if isinstance(object_type, list) else [object_type]
            if "NewsArticle" in types or "Article" in types:
                return obj
    return {}


def extract_article(candidate):
    response = session.get(candidate["url"], timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    jsonld = get_newsarticle_jsonld(soup)

    title = clean_text(jsonld.get("headline")) or candidate["title"]
    published_at = jsonld.get("datePublished") or ""
    if not published_at:
        pubdate = soup.select_one('meta[name="pubdate"]')
        published_at = pubdate.get("content", "") if pubdate else candidate["published_at_search"]

    body = clean_text(jsonld.get("articleBody"))
    if not body:
        body_root = soup.select_one(".article_body")
        if body_root:
            # 配信元名などの短い段落を避け、最も長い本文段落を採用。
            paragraph_texts = [clean_text(p.get_text("\n", strip=True)) for p in body_root.find_all("p")]
            body = max(paragraph_texts, key=len, default="")

    body_is_excerpt = not bool(body)
    if body_is_excerpt:
        body = candidate["snippet"] or clean_text(jsonld.get("description"))

    return {
        "category": candidate.get("category", ""),
        "source": candidate.get("source", ""),
        "title": title,
        "body": body,
        "published_at": published_at,
        "url": canonical_yahoo_url(response.url),
        "body_is_excerpt": body_is_excerpt,
    }


def published_datetime(record):
    try:
        value = datetime.fromisoformat(record["published_at"].replace("Z", "+00:00"))
        return value.replace(tzinfo=JST) if value.tzinfo is None else value.astimezone(JST)
    except (TypeError, ValueError, AttributeError):
        return None


In [5]:
def read_master_state(master_path):
    workbook = load_workbook(master_path, read_only=True, data_only=False)
    worksheet = workbook[workbook.sheetnames[0]]
    headers = [cell.value for cell in worksheet[1]]
    required_columns = {"ID", "category", "source", "title", "body", "published_at", "url"}
    missing_columns = required_columns - set(headers)
    if missing_columns:
        workbook.close()
        raise ValueError(f"master Excelに必要な列がありません: {sorted(missing_columns)}")

    url_index = headers.index("url") + 1
    existing_urls = {
        canonical_yahoo_url(str(row[url_index - 1]))
        for row in worksheet.iter_rows(min_row=2, values_only=True)
        if row[url_index - 1] not in (None, "")
    }
    row_count = max(worksheet.max_row - 1, 0)
    sheet_name = worksheet.title
    workbook.close()
    return sheet_name, headers, existing_urls, row_count


def append_records_to_master(master_path, sheet_name, records):
    """新規URLだけを末尾へ追加し、一時ファイル経由でmasterを置換する。"""
    if not records:
        return pd.DataFrame()

    workbook = load_workbook(master_path)
    worksheet = workbook[sheet_name]
    headers = [cell.value for cell in worksheet[1]]
    header_positions = {header: index for index, header in enumerate(headers, start=1)}
    url_column = header_positions["url"]
    id_column = header_positions["ID"]
    existing_urls = set()
    numeric_ids = []
    for row in worksheet.iter_rows(min_row=2, values_only=True):
        url_value = row[url_column - 1]
        if url_value not in (None, ""):
            existing_urls.add(canonical_yahoo_url(str(url_value)))
        id_value = row[id_column - 1]
        if str(id_value or "").isdigit():
            numeric_ids.append(int(id_value))
    next_id = max(numeric_ids, default=0) + 1
    style_source_row = worksheet.max_row if worksheet.max_row >= 2 else None
    appended_rows = []

    for record in records:
        canonical_url = canonical_yahoo_url(record.get("url", ""))
        if not canonical_url or canonical_url in existing_urls:
            continue
        row_data = {**record, "ID": next_id, "url": canonical_url}
        worksheet.append([row_data.get(header) for header in headers])
        target_row = worksheet.max_row
        if style_source_row is not None:
            for column in range(1, worksheet.max_column + 1):
                source_cell = worksheet.cell(style_source_row, column)
                target_cell = worksheet.cell(target_row, column)
                if source_cell.has_style:
                    target_cell._style = copy(source_cell._style)
        appended_rows.append({header: row_data.get(header) for header in headers})
        existing_urls.add(canonical_url)
        next_id += 1

    if not appended_rows:
        workbook.close()
        return pd.DataFrame(columns=headers)

    worksheet.auto_filter.ref = (
        f"A1:{get_column_letter(worksheet.max_column)}{worksheet.max_row}"
    )
    temporary_path = master_path.with_name(f".{master_path.stem}.updating.xlsx")
    try:
        try:
            workbook.save(temporary_path)
        finally:
            workbook.close()
        temporary_path.replace(master_path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()
    return pd.DataFrame(appended_rows, columns=headers)


master_sheet_name, master_headers, master_urls, master_row_count = read_master_state(MASTER_PATH)
print(f"master既存件数: {master_row_count:,}件 / URL: {len(master_urls):,}件")

candidates, total_results = fetch_search_candidates(KEYWORD, CUTOFF)
candidates.sort(key=lambda item: item.get("published_at_search", ""), reverse=True)
new_candidates = [
    candidate for candidate in candidates
    if canonical_yahoo_url(candidate["url"]) not in master_urls
][:MAX_NEW_ARTICLES]
print(f"\n検索総ヒット数: {total_results:,}件")
print(f"期間内のURL重複除外済み候補: {len(candidates):,}件")
print(f"master未登録・今回の取得対象: {len(new_candidates):,}件")

records = []
failures = []
seen_record_urls = set(master_urls)
for index, candidate in enumerate(new_candidates, start=1):
    record = None
    try:
        fetched_record = extract_article(candidate)
        exact_published = published_datetime(fetched_record)
        # 記事ページ側の正確な公開日時で最終判定する。
        if exact_published is None or exact_published >= CUTOFF:
            record = fetched_record
    except Exception as exc:
        # ページが削除済み等の場合も、検索結果の抜粋を残す。
        record = {
            "category": candidate.get("category", ""),
            "source": candidate.get("source", ""),
            "title": candidate["title"],
            "body": candidate["snippet"],
            "published_at": candidate["published_at_search"],
            "url": candidate["url"],
            "body_is_excerpt": True,
        }
        failures.append({"url": candidate["url"], "error": repr(exc)})

    if record is not None:
        record_url = canonical_yahoo_url(record.get("url", ""))
        # リダイレクト後のURLでも、masterおよび今回取得分との重複を除外する。
        if record_url and record_url not in seen_record_urls:
            record["url"] = record_url
            records.append(record)
            seen_record_urls.add(record_url)

    if index % 10 == 0 or index == len(new_candidates):
        print(f"記事取得: {index:,}/{len(new_candidates):,}件")
    time.sleep(REQUEST_INTERVAL_SECONDS)

articles_df = (
    append_records_to_master(MASTER_PATH, master_sheet_name, records)
    if records else pd.DataFrame(columns=master_headers)
)

print(f"\nmaster更新完了: {MASTER_PATH.resolve()}")
print(f"追加件数: {len(articles_df):,}件")
print(f"更新後件数: {master_row_count + len(articles_df):,}件")
if records:
    print(f"本文が抜粋の件数: {sum(bool(record.get('body_is_excerpt')) for record in records):,}件")
if failures:
    print(f"ページ取得失敗: {len(failures):,}件（検索結果の抜粋を追加）")

articles_df.head(10)


master既存件数: 4,692件 / URL: 4,692件

===== 国内 の検索を開始 =====
[国内] 1〜60 / ヒット 2,486件、期間内追加 50件
[国内] 51〜100 / ヒット 2,486件、期間内追加 100件
[国内] 101〜150 / ヒット 2,486件、期間内追加 150件
[国内] 151〜200 / ヒット 2,486件、期間内追加 200件
[国内] 201〜250 / ヒット 2,486件、期間内追加 250件
[国内] 251〜300 / ヒット 2,486件、期間内追加 300件
[国内] 301〜350 / ヒット 2,486件、期間内追加 350件
[国内] 351〜400 / ヒット 2,486件、期間内追加 400件
[国内] 401〜450 / ヒット 2,486件、期間内追加 450件
[国内] 451〜500 / ヒット 2,486件、期間内追加 500件
[国内] 501〜550 / ヒット 2,486件、期間内追加 550件
[国内] 551〜600 / ヒット 2,486件、期間内追加 600件
[国内] 601〜650 / ヒット 2,486件、期間内追加 650件
[国内] 651〜700 / ヒット 2,486件、期間内追加 700件
[国内] 701〜750 / ヒット 2,486件、期間内追加 750件
[国内] 751〜800 / ヒット 2,486件、期間内追加 800件
[国内] 801〜850 / ヒット 2,486件、期間内追加 850件
[国内] 851〜900 / ヒット 2,486件、期間内追加 900件
[国内] 901〜950 / ヒット 2,486件、期間内追加 950件
[国内] 951〜1,000 / ヒット 2,486件、期間内追加 1,000件
[国内] Yahoo側の1,000件上限に到達しました。

===== 国際 の検索を開始 =====
[国際] 1〜60 / ヒット 106件、期間内追加 50件
[国際] 51〜100 / ヒット 106件、期間内追加 97件

===== 経済 の検索を開始 =====
[経済] 1〜60 / ヒット 520件、期間内追加 50件
[経済] 51〜100 / ヒット 520件、期間内追加 100件
[

,カテゴリー,検索ヒット数,確認件数,期間内追加件数,期間境界到達,1000件上限到達
0,国内,2486,1000,1000,False,True
1,国際,106,100,97,True,False
2,経済,520,500,456,True,False
3,エンタメ,719,700,652,True,False
4,スポーツ,663,600,586,True,False
5,IT,153,150,117,True,False
6,科学,28,28,13,True,False
7,ライフ,248,150,131,True,False
8,地域,2036,1000,1000,False,True



検索総ヒット数: 6,959件
期間内のURL重複除外済み候補: 4,052件
master未登録・今回の取得対象: 1,000件
記事取得: 10/1,000件
記事取得: 20/1,000件
記事取得: 30/1,000件
記事取得: 40/1,000件
記事取得: 50/1,000件
記事取得: 60/1,000件
記事取得: 70/1,000件
記事取得: 130/1,000件
記事取得: 140/1,000件
記事取得: 150/1,000件
記事取得: 160/1,000件
記事取得: 170/1,000件
記事取得: 180/1,000件
記事取得: 190/1,000件
記事取得: 200/1,000件
記事取得: 210/1,000件
記事取得: 220/1,000件
記事取得: 230/1,000件
記事取得: 240/1,000件
記事取得: 250/1,000件
記事取得: 260/1,000件
記事取得: 270/1,000件
記事取得: 280/1,000件
記事取得: 290/1,000件
記事取得: 300/1,000件
記事取得: 310/1,000件
記事取得: 320/1,000件
記事取得: 330/1,000件
記事取得: 340/1,000件
記事取得: 350/1,000件
記事取得: 360/1,000件
記事取得: 370/1,000件
記事取得: 380/1,000件
記事取得: 390/1,000件
記事取得: 400/1,000件
記事取得: 410/1,000件
記事取得: 420/1,000件
記事取得: 430/1,000件
記事取得: 440/1,000件
記事取得: 450/1,000件
記事取得: 460/1,000件
記事取得: 470/1,000件
記事取得: 480/1,000件
記事取得: 490/1,000件
記事取得: 500/1,000件
記事取得: 510/1,000件
記事取得: 520/1,000件
記事取得: 530/1,000件
記事取得: 540/1,000件
記事取得: 550/1,000件
記事取得: 560/1,000件
記事取得: 570/1,000件
記事取得: 580/1,000件
記事取得: 590/1,000件
記事取得: 600/1,000件
記事取得:

,ID,category,source,title,body,published_at,url
0,4693,スポーツ,スポニチアネックス,阪神が熊本地震募金活動開催 8日、9日に京セラドームで（スポニチアネックス）,阪神は8日、9日と京セラドームで行う中日戦の試合前に、球団、選手会、OB会による「令和8年熊...,2026-08-07T15:09:36+09:00,https://news.yahoo.co.jp/articles/bf3311ec4095...
1,4694,国内,テレビ朝日系（ANN）,熊本地震を激甚災害に指定 政府「特定非常災害」にも指定（テレビ朝日系（ANN））,熊本地震を「激甚災害」に指定することを決定しました。\n\n【木原官房長官】\n「被災自治体...,2026-08-07T15:05:48+09:00,https://news.yahoo.co.jp/articles/c6b44bc9d24c...
2,4695,地域,TUFテレビユー福島,「2次被害、3次被害の可能性も…」福島赤十字病院の医師や看護師など熊本へ派遣 福島（TUFテ...,熊本地震の支援を行うため、福島市の福島赤十字病院の職員が7日、熊本県に派遣されました。\n【...,2026-08-07T15:02:14+09:00,https://news.yahoo.co.jp/articles/c6426338a54b...
3,4696,IT,Web担当者Forum,「Studio」有料プランの災害時の無償提供に関する相談窓口「災害支援相談フォーム」を開設（...,AI搭載のウェブ制作プラットフォーム「Studio(スタジオ)」を展開するStudioは、プ...,2026-08-07T15:01:34+09:00,https://news.yahoo.co.jp/articles/77c28b89c85b...
4,4697,国内,時事通信,【熊本地震】ボランティアは「自己完結」で◆準備やけが防止、熱中症対策を（時事通信）,被災地に入る前に準備すべきものは何か。被災地に負担を掛けない「自己完結」が原則で、それぞれ現...,2026-08-07T15:00:50+09:00,https://news.yahoo.co.jp/articles/6d4c43c38afb...
5,4698,エンタメ,モデルプレス,HIKAKIN、2000万円を寄付 会社・個人から1000万＆1万4400本の飲料を被災地へ...,【モデルプレス＝2026/08/07】YouTuberのHIKAKINが8月7日、自身のYo...,2026-08-07T14:59:10+09:00,https://news.yahoo.co.jp/articles/34e4778d4a59...
6,4699,エンタメ,テレビ朝日系（ANN）,浴場＆大広間を無料開放中のスザンヌ「この数日、本当にたくさんの優しさに触れています」（テレビ...,令和8年熊本地震を受け、熊本市内で経営する旅館の浴場と大広間の無料開放を行っているタレント・...,2026-08-07T14:57:14+09:00,https://news.yahoo.co.jp/articles/050d4d3fe4bf...
7,4700,スポーツ,スポーツ報知,【甲子園】熊本代表の有明・高見監督「僕たちができることは本当にちっちゃなこと」朝にはコインラ...,◆第１０８回全国高校野球選手権大会第３日 ▽１回戦 立命館宇治―有明（７日・甲子園）\n【夏...,2026-08-07T14:54:36+09:00,https://news.yahoo.co.jp/articles/bff8d826a053...
8,4701,国内,FNNプライムオンライン,"「語らないまま死なないで」81歳の被爆者が始めた""最後の記録"" 鹿児島・伊佐市（FNNプライ...",80歳を過ぎた今、西上床さんは新たな取り組みを始めた。県内に残る被爆者の声を2年間をめどに聞...,2026-08-07T14:53:20+09:00,https://news.yahoo.co.jp/articles/8129c82e2f42...
9,4702,国内,時事通信,政府、熊本地震を激甚災害に指定 特定非常災害にも（時事通信）,政府は7日の持ち回り閣議で、熊本地震を激甚災害と特定非常災害に指定することを決めた。,2026-08-07T14:47:31+09:00,https://news.yahoo.co.jp/articles/c615d8fe994b...


In [6]:
articles_df.tail(10)

,ID,category,source,title,body,published_at,url
990,5683,国内,読売新聞オンライン,熊本地震1週間 愛知県が支援対策本部…医療チームら派遣本格化（読売新聞オンライン）,熊本県で最大震度7を観測した地震の発生から1週間となった4日、愛知県内では医師や看護師、自治...,2026-08-05T15:23:37+09:00,https://news.yahoo.co.jp/articles/3260541baec8...
991,5684,経済,産経新聞,トヨタ、日産の九州工場が6日に再開 熊本地震で停止の拠点に再稼働の動き広がる（産経新聞）,熊本地震の影響で停止していた自動車大手の工場に生産再開の動きが広がってきた。トヨタ自動車は5...,2026-08-05T15:20:36+09:00,https://news.yahoo.co.jp/articles/f263406e8343...
992,5685,国内,共同通信,業績影響を精査中と日本製紙社長（共同通信）,熊本地震で9人が死亡した日本製紙の瀬辺社長は「業績への影響を精査している」と話し、判明すれば...,2026-08-05T15:17:52+09:00,https://news.yahoo.co.jp/articles/d783b4b466cd...
993,5686,スポーツ,スポニチアネックス,ラグビー日本代表・姫野和樹 熊本地震の被災地へ300万円寄付「ワンチームで乗り越えていきまし...,ラグビー日本代表の姫野和樹（32）が熊本地震の被災地へ300万円を寄付したことを5日までに自...,2026-08-05T15:16:39+09:00,https://news.yahoo.co.jp/articles/b9ee60272848...
994,5687,国内,共同通信,イオンモール熊本に献花台 爆発犠牲者悼み、復旧願う（共同通信）,イオンモールは5日、爆発の起きた「イオンモール熊本」（熊本県嘉島町）の中央出入り口付近に献花...,2026-08-05T15:15:42+09:00,https://news.yahoo.co.jp/articles/7130c2bb4c1b...
995,5688,地域,RKK熊本放送,【速報】「本当に悲しく残念な事態」日本製紙が会見 熊本地震で煙突倒壊 9人死亡 「前回の熊本...,熊本地震で煙突が倒壊し、9人が死亡した日本製紙が午後3時から会見を開きました。以下、会見の内...,2026-08-05T15:14:13+09:00,https://news.yahoo.co.jp/articles/c1547137f7ac...
996,5689,経済,読売新聞オンライン,九州新幹線の新水俣―鹿児島中央駅間の運行を７日に再開…全て自由席で各駅に停車、上下線各６本（...,ＪＲ九州は５日、熊本地震の影響で運転を取りやめている九州新幹線の新水俣―鹿児島中央駅間の運行...,2026-08-05T15:11:41+09:00,https://news.yahoo.co.jp/articles/4b3b4fe13b2c...
997,5690,国内,共同通信,「深くおわび」と日本製紙社長（共同通信）,熊本地震で日本製紙八代工場の煙突が折れ9人が死亡した事故を巡り、日本製紙の瀬辺明社長は5日、...,2026-08-05T15:10:23+09:00,https://news.yahoo.co.jp/articles/7c41593c96ee...
998,5691,地域,熊本日日新聞,【速報】日本製紙社長、被害者に「お悔やみ申し上げる」 地震後、初の会見（熊本日日新聞）,2026年熊本地震で煙突が倒壊した八代市の日本製紙八代工場で、従業員ら9人が亡くなったことを...,2026-08-05T15:09:21+09:00,https://news.yahoo.co.jp/articles/b6bf0942ce6f...
999,5692,IT,いしたにまさき,「デジタル面での被災地支援」デジタル庁がガバメントAI「源内」を緊急提供 #エキスパートトピ...,デジタル庁が、政府職員向け生成AI「ガバメントAI 源内」を令和8年熊本地震の被災自治体や災...,2026-08-05T15:05:12+09:00,https://news.yahoo.co.jp/expert/articles/c1657...


In [ ]:
master_df = pd.read_excel(MASTER_PATH)

In [ ]:
master_df.tail(10)